In [0]:
%pip install --upgrade pydantic
%pip uninstall -y evidently
%pip install evidently
%pip install --upgrade evidently

In [0]:
%restart_python


## ¿Qué es MLflow?

MLflow es una plataforma de código abierto para gestionar el ciclo de vida de modelos de machine learning. Permite registrar experimentos, guardar parámetros, métricas, artefactos y modelos, facilitando la trazabilidad y comparación entre diferentes ejecuciones.

### Componentes principales de MLflow

- **Tracking:** Registra experimentos, parámetros, métricas y artefactos. Permite comparar resultados y visualizar el progreso de los modelos.
- **Projects:** Estandariza la ejecución de proyectos de ML, definiendo entornos y dependencias.
- **Models:** Permite guardar, cargar y desplegar modelos en diferentes formatos y entornos.
- **Registry:** Gestiona versiones de modelos, facilitando su promoción a producción y control de acceso.

### Ventajas de MLflow

- Facilita la experimentación y comparación de modelos.
- Permite reproducir resultados y compartir experimentos.
- Integración sencilla con frameworks populares (scikit-learn, TensorFlow, PyTorch, etc.).
- Soporte para visualización de métricas y artefactos.

MLflow es ideal para mantener un registro organizado de experimentos, mejorar la colaboración y acelerar el desarrollo de modelos de machine learning.

In [0]:
# kDescarga dataset de ejemplo
from sklearn.datasets import load_breast_cancer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import mlflow
import mlflow.sklearn

# Carga y prepara datos
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

# Define hiperparámetros para experimentar
param_grid = [
    {"n_estimators": 50, "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 150, "max_depth": 7},
    {"n_estimators": 200, "max_depth": 9}
]

mlflow.set_experiment("/Users/mblancovillar@itba.edu.ar/Clase Prueba")

for params in param_grid:
    with mlflow.start_run():
        clf = RandomForestClassifier(n_estimators=params["n_estimators"], max_depth=params["max_depth"], random_state=42)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_proba = clf.predict_proba(X_test)[:, 1]
        acc = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)
        
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", auc)
        #mlflow.sklearn.log_model(clf, "model")

In [0]:
mlflow.sklearn.log_model(clf, name="model")

In [0]:
# Descarga dataset de ejemplo
from sklearn.datasets import load_breast_cancer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import pydantic

# Carga y prepara datos
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

# Define hiperparámetros para experimentar
param_grid = [
    {"n_estimators": 50, "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 150, "max_depth": 7},
    {"n_estimators": 200, "max_depth": 9}
]

mlflow.set_experiment("/Users/mblancovillar@itba.edu.ar/Clase Prueba")

best_auc = 0
best_run_id = None

for params in param_grid:
    with mlflow.start_run() as run:
        clf = RandomForestClassifier(n_estimators=params["n_estimators"], max_depth=params["max_depth"], random_state=42)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_proba = clf.predict_proba(X_test)[:, 1]
        acc = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)
        cm = confusion_matrix(y_test, y_pred)
        report = classification_report(y_test, y_pred, output_dict=True)
        signature = infer_signature(X_test, y_pred)
        
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", auc)
        mlflow.log_metric("precision", report["1"]["precision"])
        mlflow.log_metric("recall", report["1"]["recall"])
        mlflow.log_metric("f1_score", report["1"]["f1-score"])
        mlflow.sklearn.log_model(clf, signature=signature)
        mlflow.log_dict(report, "classification_report.json")
        mlflow.log_dict({"confusion_matrix": cm.tolist()}, "confusion_matrix.json")
        
        if auc > best_auc:
            best_auc = auc
            best_run_id = run.info.run_id

In [0]:
# Descarga dataset de ejemplo
from sklearn.datasets import load_breast_cancer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import pydantic

# Carga y prepara datos
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

# Define hiperparámetros para experimentar
param_grid = [
    {"n_estimators": 50, "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 150, "max_depth": 7},
    {"n_estimators": 200, "max_depth": 9}
]

mlflow.set_experiment("/Users/mblancovillar@itba.edu.ar/Clase Prueba")

best_auc = 0
best_run_id = None

with mlflow.start_run(run_name="parent_run") as parent_run:
    mlflow.log_param("experiment_type", "RandomForest_Hyperparameter_Search")
    for params in param_grid:
        with mlflow.start_run(run_name=f"child_run_{params['n_estimators']}_{params['max_depth']}", nested=True):
            clf = RandomForestClassifier(n_estimators=params["n_estimators"], max_depth=params["max_depth"], random_state=42)
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            y_proba = clf.predict_proba(X_test)[:, 1]
            acc = accuracy_score(y_test, y_pred)
            auc = roc_auc_score(y_test, y_proba)
            cm = confusion_matrix(y_test, y_pred)
            report = classification_report(y_test, y_pred, output_dict=True)
            signature = infer_signature(X_test, y_pred)
            
            mlflow.log_params(params)
            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("roc_auc", auc)
            mlflow.log_metric("precision", report["1"]["precision"])
            mlflow.log_metric("recall", report["1"]["recall"])
            mlflow.log_metric("f1_score", report["1"]["f1-score"])
            mlflow.sklearn.log_model(clf, signature=signature)
            mlflow.log_dict(report, "classification_report.json")
            mlflow.log_dict({"confusion_matrix": cm.tolist()}, "confusion_matrix.json")
            
            if auc > best_auc:
                best_auc = auc
                best_run_id = mlflow.active_run().info.run_id

## ¿Qué es Optuna?

Optuna es un framework de optimización automática de hiperparámetros para machine learning y ciencia de datos. Utiliza algoritmos avanzados para buscar eficientemente la mejor configuración de parámetros de modelos, adaptándose dinámicamente a los resultados obtenidos.

### Optimizadores disponibles en Optuna

Optuna soporta varios algoritmos de optimización, llamados "samplers":

- **TPE (Tree-structured Parzen Estimator):** Algoritmo bayesiano que modela la probabilidad de buenos y malos resultados, recomendando parámetros prometedores. Es el sampler por defecto (`optuna.samplers.TPESampler`).
- **Random Search:** Selecciona combinaciones de parámetros al azar. Útil como baseline o para problemas simples (`optuna.samplers.RandomSampler`).
- **CMA-ES (Covariance Matrix Adaptation Evolution Strategy):** Algoritmo evolutivo para optimización continua, especialmente útil en espacios de parámetros complejos (`optuna.samplers.CmaEsSampler`).
- **Grid Search:** Prueba todas las combinaciones posibles de parámetros definidos en una cuadrícula. No es eficiente para espacios grandes (`optuna.samplers.GridSampler`).
- **Brute Force:** Similar a grid search, pero explora exhaustivamente el espacio de búsqueda.
- **Sobol y Halton:** Métodos de muestreo cuasi-aleatorio para cubrir el espacio de búsqueda de manera uniforme (`optuna.samplers.SobolSampler`, `optuna.samplers.HaltonSampler`).

Optuna también permite definir restricciones, condiciones, y estrategias de parada temprana ("pruners") para acelerar la búsqueda y evitar pruebas innecesarias.

### Ventajas de Optuna

- Búsqueda eficiente y adaptativa.
- Fácil integración con frameworks de ML.
- Soporte para visualización y análisis de resultados.
- Permite optimización multiobjetivo y paralelización.

Optuna es ideal para encontrar automáticamente la mejor configuración de modelos, ahorrando tiempo y recursos en experimentación.



In [0]:
%pip install optuna
import optuna
from sklearn.datasets import load_breast_cancer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

# Carga y prepara datos
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

mlflow.set_experiment("/Users/mblancovillar@itba.edu.ar/Clase Prueba")

def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 200, step=50)
    max_depth = trial.suggest_int("max_depth", 3, 9, step=2)
    clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    trial.set_user_attr("clf", clf)
    trial.set_user_attr("y_pred", y_pred)
    trial.set_user_attr("y_proba", y_proba)
    return auc

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

# Log only the most promising trials (e.g., top 3)
top_trials = sorted(study.trials, key=lambda t: t.value, reverse=True)[:3]

for trial in top_trials:
    clf = trial.user_attrs["clf"]
    y_pred = trial.user_attrs["y_pred"]
    y_proba = trial.user_attrs["y_proba"]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    signature = infer_signature(X_test, y_pred)
    params = trial.params
    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}"):
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", auc)
        mlflow.log_metric("precision", report["1"]["precision"])
        mlflow.log_metric("recall", report["1"]["recall"])
        mlflow.log_metric("f1_score", report["1"]["f1-score"])
        mlflow.sklearn.log_model(clf, signature=signature)
        mlflow.log_dict(report, "classification_report.json")
        mlflow.log_dict({"confusion_matrix": cm.tolist()}, "confusion_matrix.json")

In [0]:
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 200, step=50)
    max_depth = trial.suggest_int("max_depth", 3, 9, step=2)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)
    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    signature = infer_signature(X_test, y_pred)
    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}"):
        mlflow.log_params({
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "min_samples_split": min_samples_split,
            "min_samples_leaf": min_samples_leaf
        })
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", auc)
        mlflow.log_metric("precision", report["1"]["precision"])
        mlflow.log_metric("recall", report["1"]["recall"])
        mlflow.log_metric("f1_score", report["1"]["f1-score"])
        mlflow.sklearn.log_model(clf, signature=signature)
        mlflow.log_dict(report, "classification_report.json")
        mlflow.log_dict({"confusion_matrix": cm.tolist()}, "confusion_matrix.json")
    trial.set_user_attr("clf", clf)
    trial.set_user_attr("y_pred", y_pred)
    trial.set_user_attr("y_proba", y_proba)
    trial.set_user_attr("acc", acc)
    trial.set_user_attr("cm", cm)
    trial.set_user_attr("report", report)
    trial.set_user_attr("signature", signature)
    return auc

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

top_trials = sorted(study.trials, key=lambda t: t.value, reverse=True)[:3]

for trial in top_trials:
    clf = trial.user_attrs["clf"]
    y_pred = trial.user_attrs["y_pred"]
    y_proba = trial.user_attrs["y_proba"]
    acc = trial.user_attrs["acc"]
    auc = roc_auc_score(y_test, y_proba)
    cm = trial.user_attrs["cm"]
    report = trial.user_attrs["report"]
    signature = trial.user_attrs["signature"]
    params = trial.params
    # Logging already handled inside objective


### Visualización de la optimización con Optuna

Optuna ofrece varias herramientas visuales para analizar el proceso de búsqueda de hiperparámetros:

- **plot_optimization_history:** Muestra cómo evoluciona el valor objetivo (por ejemplo, ROC AUC) a lo largo de los ensayos, permitiendo ver la mejora progresiva del modelo.
- **plot_param_importances:** Indica qué hiperparámetros tienen mayor impacto en el resultado, ayudando a identificar los más relevantes para el modelo.
- **plot_parallel_coordinate:** Permite explorar cómo diferentes combinaciones de parámetros afectan el desempeño, visualizando relaciones entre ellos.
- **plot_slice:** Analiza el efecto de cada parámetro individual sobre el valor objetivo, mostrando cómo varía el resultado según su valor.

Estas visualizaciones facilitan la interpretación de la búsqueda, ayudan a seleccionar los mejores modelos y guían futuras optimizaciones de manera eficiente y transparente.

In [0]:
import optuna.visualization

# Visualización de la optimización con Optuna
optuna.visualization.plot_optimization_history(study)
optuna.visualization.plot_param_importances(study)
optuna.visualization.plot_parallel_coordinate(study)
optuna.visualization.plot_slice(study)


### Conclusiones sobre la visualización de la optimización

El gráfico de la historia de optimización muestra cómo Optuna explora diferentes combinaciones de hiperparámetros y mejora progresivamente el valor de ROC AUC. Se observa que, a medida que avanzan los ensayos, el algoritmo encuentra configuraciones más efectivas, lo que evidencia la eficiencia de la búsqueda bayesiana frente a métodos tradicionales. Además, la visualización de la importancia de los parámetros permite identificar cuáles tienen mayor impacto en el desempeño del modelo, guiando futuras optimizaciones y análisis. En conjunto, estas herramientas facilitan la toma de decisiones y la selección de los mejores modelos de manera automatizada y transparente.

## Evidently: Monitoreo de Drift y Calidad de Datos

Evidently es una herramienta open source para monitoreo de modelos de machine learning y análisis de calidad de datos. Permite detectar cambios ("drift") en la distribución de datos, evaluar el desempeño de modelos en producción y generar reportes automáticos sobre métricas clave.

### Funcionalidades principales

- **Detección de Data Drift:** Identifica si la distribución de las variables de entrada cambia entre el dataset de referencia y el actual, lo que puede afectar la performance del modelo.
- **Monitoreo de Calidad de Datos:** Evalúa la presencia de valores nulos, outliers, y otras anomalías en los datos.
- **Evaluación de Modelos:** Analiza métricas de clasificación/regresión, como precisión, recall, ROC AUC, y compara el desempeño entre datasets.
- **Reportes Interactivos:** Genera dashboards y reportes visuales para facilitar la interpretación y el seguimiento de los resultados.

### Ejemplo de uso

En el flujo actual, Evidently se utiliza para comparar el dataset de test (actual) con el de entrenamiento (referencia), detectando posibles drifts y evaluando la calidad de las predicciones del modelo Random Forest. Esto ayuda a anticipar problemas antes de que impacten en producción y a mantener la robustez del sistema de ML.

In [0]:
import numpy as np
# Restaurar el alias eliminado para compatibilidad
np.float_ = np.float64 

from evidently import Report
from evidently.presets import DataDriftPreset, ClassificationPreset


# Evidently analysis for the last trained model
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

# Ensure prediction and proba have correct lengths
data_current = pd.DataFrame(X_test).assign(target=y_test)
if len(y_pred) == len(y_test):
    data_current["prediction"] = y_pred
if len(y_proba) == len(y_test):
    data_current["proba"] = y_proba

report = Report(metrics=[
    DataDriftPreset(),
    ClassificationPreset()
])



In [0]:
data_current

In [0]:
y_train_pred = clf.predict(X_train)
data_reference = pd.DataFrame(X_train).assign(target=y_train, prediction=y_train_pred)


In [0]:
y_train_proba = clf.predict_proba(X_train)[:, 1]
data_reference["proba"] = y_train_proba


In [0]:
data_reference

In [0]:
from evidently.future.datasets import Dataset, DataDefinition
from evidently import MulticlassClassification

data_definition = DataDefinition(
    classification=[MulticlassClassification(
        target="target",
        prediction_labels="prediction",
        prediction_probas="proba",
    )]
)

current_dataset = Dataset.from_pandas(data_current, data_definition=data_definition)
reference_dataset = Dataset.from_pandas(data_reference, data_definition=data_definition)

In [0]:
report.run(
    current_data=current_dataset,
    reference_data=reference_dataset
)

#  Conclusiones del reporte de Evidently

---

## 1. Dataset Drift: prácticamente inexistente
- Solo **1 de 33 columnas** con drift  
- Share drifted: **0.03 (~3%)**  
- El reporte indica: *Dataset Drift is NOT detected*

**Conclusión:**  
✔️ Los datos en producción son **muy similares a entrenamiento**  
✔️ No hay problema de distribución (por ahora)

---

## 2. Performance del modelo: sigue siendo muy alta

### Modelo actual:
- Accuracy: **0.958**
- Precision: **0.957**
- Recall: **0.978**
- F1: **0.967**
- ROC AUC: **0.995**

**Conclusión:**  
✔️ El modelo sigue funcionando **excelente en producción**

---

## 3. Leve degradación vs modelo de referencia

### Modelo de referencia:
- Métricas: **1.0 (perfecto)**

### Modelo actual:
- Leve caída en todas las métricas

**Conclusión importante:**  
No hay drift fuerte, pero **sí hay degradación leve del modelo**

Posibles causas:
- ruido en datos reales  
- diferencias no capturadas como drift estadístico  

---

## 4. Confusion Matrix: errores controlados
- Muy pocos falsos positivos / negativos  
- Clasificación estable  

**Conclusión:**  
✔️ No hay problema operativo crítico  

---

## 5. Precision-Recall Curve estable
- Curva cercana al óptimo  
- No hay colapso del modelo  

**Conclusión:**  
✔️ Buen balance entre precisión y recall  

---




## ¿Qué es SHAP?

SHAP (SHapley Additive exPlanations) es una técnica de interpretabilidad para modelos de machine learning basada en la teoría de juegos. SHAP calcula la contribución de cada característica a la predicción de un modelo, permitiendo entender cómo y por qué el modelo toma decisiones.

### Principales ventajas de SHAP

- **Atribución precisa:** Asigna a cada feature un valor que representa su impacto en la predicción.
- **Consistencia:** Los valores SHAP reflejan el cambio esperado en la predicción si se modifica una característica.
- **Visualizaciones intuitivas:** Permite analizar la importancia global y local de las variables, identificar patrones y detectar posibles sesgos.

### Ejemplo de uso

En este flujo, SHAP se utiliza para analizar el modelo Random Forest entrenado, mostrando:
- Qué variables influyen más en la predicción.
- Cómo cada característica afecta la decisión para casos individuales.
- Visualizaciones como summary plot y force plot para interpretar el modelo.

SHAP ayuda a validar la lógica del modelo, aumentar la confianza y detectar posibles problemas de interpretabilidad.

In [0]:
#
%pip install shap
import shap

# SHAP analysis for the last trained model
shap.initjs()
explainer = shap.Explainer(clf, X_train)
shap_values = explainer(X_test)

# SHAP summary plot
shap.summary_plot(shap_values.values, X_test, feature_names=X_test.columns)

In [0]:
shap_values


In [0]:
shap.plots.waterfall(shap_values[0, :, 1])

In [0]:
shap.plots.beeswarm(shap_values[:, :, 1])

In [0]:

shap.plots.force(
    base_value=explainer.expected_value[1],
    shap_values=shap_values.values[1],
 )

In [0]:
importances=clf.feature_importances_
feature_names=data.feature_names

In [0]:
import matplotlib.pyplot as plt
sorted_idx = np.argsort(importances)[::-1]
plt.barh(np.array(feature_names)[sorted_idx], importances[sorted_idx])
plt.xlabel("Feature Importance")
plt.title("Feature Importance in Random Forest Classifier")
plt.gca().invert_yaxis()
plt.show()